# MIT License

Copyright (c) 2024 Social Science Research Notebooks

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Running in Colab')

# Time Series Analysis In-Depth

Analyzing trends and seasonality in time-series data using classical statistical methods.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
import warnings
warnings.filterwarnings('ignore')

# Generate Synthetic Time Series Data
dates = pd.date_range(start='2020-01-01', periods=100, freq='ME')
trend = np.linspace(0, 50, 100)
seasonality = 10 * np.sin(np.linspace(0, 3.14 * 10, 100))
noise = np.random.normal(0, 2, 100)
values = trend + seasonality + noise

df = pd.DataFrame({'Date': dates, 'Value': values})
df.set_index('Date', inplace=True)

df.plot(figsize=(10, 5))
plt.title('Synthetic Time Series Data')
plt.show()

In [ ]:
# Decomposition
decomposition = seasonal_decompose(df['Value'], model='additive')

fig = decomposition.plot()
fig.set_size_inches(10, 8)
plt.show()

## 3. Stationarity Check (Augmented Dickey-Fuller Test)

Before applying ARIMA, we must check if the series is stationary. If the p-value > 0.05, the data is non-stationary and needs differencing.

In [ ]:
result = adfuller(df['Value'])
print('ADF Statistic:', result[0])
print('p-value:', result[1])

if result[1] > 0.05:
    print("Series is likely non-stationary.")
else:
    print("Series is likely stationary.")

## 4. Autocorrelation Plots (ACF & PACF)

These plots help determine the order of the AR (p) and MA (q) terms. We look at the differenced series if the original was non-stationary.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 4))

# First difference to make it stationary
df_diff = df['Value'].diff().dropna()

plot_acf(df_diff, ax=ax[0])
plot_pacf(df_diff, ax=ax[1])
plt.show()

## 5. ARIMA Modeling

We fit an ARIMA(1,1,1) model as a starting point based on the plots (1 lag differencing).

In [ ]:
# Fit ARIMA model (p=1, d=1, q=1)
model = ARIMA(df['Value'], order=(1, 1, 1))
model_fit = model.fit()
print(model_fit.summary())

In [ ]:
# Forecast 12 months ahead
forecast_res = model_fit.get_forecast(steps=12)
forecast_val = forecast_res.predicted_mean
conf_int = forecast_res.conf_int()

plt.figure(figsize=(10, 5))
plt.plot(df.index, df['Value'], label='Observed')
plt.plot(pd.date_range(df.index[-1], periods=13, freq='ME')[1:], forecast_val, label='Forecast', color='red')
plt.fill_between(pd.date_range(df.index[-1], periods=13, freq='ME')[1:], 
                 conf_int.iloc[:, 0], conf_int.iloc[:, 1], color='pink', alpha=0.3)
plt.legend()
plt.title('ARIMA Forecast')
plt.show()